In [2]:
import os
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision import models

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

if torch.backends.mps.is_available(): device = torch.device("mps")
elif torch.cuda.is_available(): device = torch.device("cuda")
else: device = torch.device("cpu")

In [3]:
model=models.efficientnet_b0(weights="DEFAULT")

for param in model.features.parameters():
    param.requires_grad=False

model.classifier[1]=nn.Linear(1280, 10)

model=model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [3]:
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(224, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]),
])

train_data = datasets.CIFAR10('/Users/hwanghyunjin/Desktop/cv-portfolio/week1/data', train=True, download=True, transform=transform)
test_data = datasets.CIFAR10('/Users/hwanghyunjin/Desktop/cv-portfolio/week1/data', train=False, transform=test_transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)

100.0%


In [4]:
def train(model, loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0
    
    for batch_idx, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    print(f'Epoch {epoch}: Loss = {total_loss/len(loader):.4f}')
    
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    correct = 0
    
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)  
            output = model(data)
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
    acc = 100. * correct / len(loader.dataset)
    
    print(f'Accuracy: {acc:.2f}%')
    
    return acc

In [6]:
train_losses = []
val_accs = []

for epoch in range(1, 4):
    loss = train(model, train_loader, criterion, optimizer, epoch)
    acc = evaluate(model, test_loader)
    #scheduler.step()
    train_losses.append(loss)
    val_accs.append(acc)

Epoch 1: Loss = 0.6888
Accuracy: 81.68%
Epoch 2: Loss = 0.6652
Accuracy: 82.11%
Epoch 3: Loss = 0.6511
Accuracy: 81.76%


In [4]:
n = sum(p.numel() for p in model.parameters())
print(f"{n/1e6:.1f}M")

4.0M
